**IMPORTS**

In [19]:
import os
import torch
import numpy as np
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, DeiTFeatureExtractor, ViTModel
from PIL import Image
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

**CONFIG**

In [28]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = "../checkpoints/early_fusion_unixcoder_nn/"
BATCH_SIZE = 8
EPOCHS = 3
LR = 2e-5

UNIXCODER_CKPT = "../checkpoints/unixcoder_only/checkpoint-2322/"
VIT_CKPT = "../checkpoints/vit_only/deit_epoch_3.pt"

TEXT_DIR = "../Text_Files/Train"
IMAGE_DIR = "../snapshots/Train"
TEST_BASE = "../snapshots"

**LOAD MODELS**

In [21]:
from transformers import ViTForImageClassification, AutoImageProcessor

print("Loading fine-tuned UnixCoder...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/unixcoder-base")
text_model = AutoModel.from_pretrained(UNIXCODER_CKPT).to(DEVICE)
text_model.eval()

Loading fine-tuned UnixCoder...


Some weights of RobertaModel were not initialized from the model checkpoint at ../checkpoints/unixcoder_only/checkpoint-2322/ and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(51416, 768, padding_idx=1)
    (position_embeddings): Embedding(1026, 768, padding_idx=1)
    (token_type_embeddings): Embedding(10, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (

In [22]:
print("Loading fine-tuned DeiT model...")
image_processor = AutoImageProcessor.from_pretrained("facebook/deit-base-patch16-224")

# Load your trained classification model first
trained_model = ViTForImageClassification.from_pretrained(
    "facebook/deit-base-patch16-224",
    num_labels=2,
    ignore_mismatched_sizes=True
)

trained_model.load_state_dict(torch.load(VIT_CKPT, map_location=DEVICE))
trained_model.to(DEVICE)

# Extract just the ViT base model for embeddings
vit_model = trained_model.vit# This gives you the pure ViTModel
vit_model.to(DEVICE)
vit_model.eval()

Loading fine-tuned DeiT model...


Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 21076.90it/s]
Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 29127.11it/s]
Some weights of ViTForImageClassification were not initialized from the model checkpoint at facebook/deit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ViTModel(
  (embeddings): ViTEmbeddings(
    (patch_embeddings): ViTPatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): ViTEncoder(
    (layer): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTAttention(
          (attention): ViTSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
          )
          (output): ViTSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (intermediate): ViTIntermediate(
          (dense): Linear(in_features=768, out_features=3072, bias=True)
          (intermediate_act_fn): GELUActivation()
        )
        (output): ViTOutput(
          (d

**DATASET**

In [23]:
class FusionDataset(Dataset):
    def __init__(self, text_dir, image_dir, tokenizer, image_processor):
        self.text_paths = []
        self.image_paths = []
        self.labels = []
        self.tokenizer = tokenizer
        self.image_processor = image_processor

        # Scan text folders
        for label_folder in sorted(os.listdir(text_dir)):
            label_path = os.path.join(text_dir, label_folder)
            if not os.path.isdir(label_path):
                continue
            label = int(label_folder.split("_")[1])  # e.g., "Label_0" -> 0
            for txt_file in sorted(os.listdir(label_path)):
                if txt_file.endswith(".txt"):
                    self.text_paths.append(os.path.join(label_path, txt_file))
                    self.labels.append(label)

        # Scan image folders
        self.image_paths = []
        for label_folder in sorted(os.listdir(image_dir)):
            label_path = os.path.join(image_dir, label_folder)
            if not os.path.isdir(label_path):
                continue
            for img_file in sorted(os.listdir(label_path)):
                if img_file.lower().endswith((".png", ".jpg", ".jpeg")):
                    self.image_paths.append(os.path.join(label_path, img_file))

        # Ensure text_paths and image_paths are aligned
        assert len(self.text_paths) == len(self.image_paths), "Text and image counts must match!"

    def __len__(self):
        return len(self.text_paths)

    def __getitem__(self, idx):
        # ----- TEXT -----
        with open(self.text_paths[idx], "r") as f:
            text = f.read()
        encoding = self.tokenizer(
            text, return_tensors="pt", truncation=True, padding="max_length", max_length=512
        )
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        # ----- IMAGE -----
        image = Image.open(self.image_paths[idx]).convert("RGB")
        image_tensor = self.image_processor(images=image, return_tensors="pt")
        for k in image_tensor:
            image_tensor[k] = image_tensor[k].squeeze(0)

        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return input_ids, attention_mask, image_tensor, label


**DATA LOADING**

In [24]:
dataset = FusionDataset(TEXT_DIR, IMAGE_DIR, tokenizer, image_processor)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

**MODEL ARCHITECTURE**

In [25]:
class FusionClassifier(nn.Module):
    def __init__(self, text_model, vit_model, hidden_dim=512, num_classes=2):
        super().__init__()
        self.text_model = text_model
        self.vit_model = vit_model

        # freeze backbone models
        for p in self.text_model.parameters():
            p.requires_grad = False
        for p in self.vit_model.parameters():
            p.requires_grad = False

        text_emb_dim = text_model.config.hidden_size
        vit_emb_dim = vit_model.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(text_emb_dim + vit_emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes)
        )

        # Add Layer Normalization for fused embeddings
        self.fused_norm = nn.LayerNorm(text_emb_dim + vit_emb_dim)

    def forward(self, input_ids, attention_mask, image_tensor):
        # Text embedding: CLS token
        text_outputs = self.text_model(input_ids=input_ids, attention_mask=attention_mask)
        text_cls = text_outputs.last_hidden_state[:, 0, :]  # [batch, hidden]

        # Image embedding: CLS token
        image_outputs = self.vit_model(**{k: v for k, v in image_tensor.items()})
        image_cls = image_outputs.last_hidden_state[:, 0, :]  # [batch, hidden]

        # Concatenate and classify
        fused = torch.cat([text_cls, image_cls], dim=1)
        fused_normalized = self.fused_norm(fused)  # Apply layer normalization

        logits = self.classifier(fused_normalized)
        return logits

**OPTIMIZERS && LOSS**

In [26]:
model = FusionClassifier(text_model, vit_model).to(DEVICE)
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

**TRAINING LOOP && CHECKPOINTING**

In [29]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader):
        input_ids, attention_mask, image_tensor, labels = batch
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        labels = labels.to(DEVICE)
        for k in image_tensor:
            image_tensor[k] = image_tensor[k].to(DEVICE)

        logits = model(input_ids, attention_mask, image_tensor)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Avg Loss: {avg_loss:.4f}")

    # Save checkpoint
    ckpt_path = os.path.join(OUTPUT_DIR, f"fusion_norm_epoch_{epoch+1}.pt")
    torch.save(model.state_dict(), ckpt_path)
    print(f"Checkpoint saved: {ckpt_path}")

100%|██████████| 774/774 [01:24<00:00,  9.14it/s]


Epoch 1/3 | Avg Loss: 0.0699
Checkpoint saved: ../checkpoints/early_fusion_unixcoder_nn/fusion_norm_epoch_1.pt


100%|██████████| 774/774 [01:25<00:00,  9.03it/s]


Epoch 2/3 | Avg Loss: 0.0675
Checkpoint saved: ../checkpoints/early_fusion_unixcoder_nn/fusion_norm_epoch_2.pt


100%|██████████| 774/774 [01:27<00:00,  8.89it/s]


Epoch 3/3 | Avg Loss: 0.0645
Checkpoint saved: ../checkpoints/early_fusion_unixcoder_nn/fusion_norm_epoch_3.pt


**FUSION EMBEDDING ANALYSIS**

In [30]:
def analyze_fusion_embeddings(text_model, vit_model, dataloader, device):
    """
    Analyze embeddings from text, image, and fused representations
    """
    text_model.eval()
    vit_model.eval()
    
    text_embeddings = []
    image_embeddings = [] 
    fused_embeddings = []
    labels_list = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extracting multi-modal embeddings"):
            input_ids, attention_mask, image_tensor, labels = batch
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)
            
            # Move image tensors to device
            for k in image_tensor:
                image_tensor[k] = image_tensor[k].to(device)
            
            # Text embeddings
            text_outputs = text_model(input_ids=input_ids, attention_mask=attention_mask)
            text_emb = text_outputs.last_hidden_state[:, 0, :].cpu().numpy()
            
            # Image embeddings
            image_outputs = vit_model(**{k: v for k, v in image_tensor.items()})
            image_emb = image_outputs.last_hidden_state[:, 0, :].cpu().numpy()
            
            # Fused embeddings (concatenation)
            fused_emb = np.concatenate([text_emb, image_emb], axis=1)
            
            text_embeddings.append(text_emb)
            image_embeddings.append(image_emb)
            fused_embeddings.append(fused_emb)
            labels_list.append(labels.cpu().numpy())
    
    text_embeddings = np.concatenate(text_embeddings, axis=0)
    image_embeddings = np.concatenate(image_embeddings, axis=0)
    fused_embeddings = np.concatenate(fused_embeddings, axis=0)
    labels = np.concatenate(labels_list, axis=0)
    
    return text_embeddings, image_embeddings, fused_embeddings, labels

def compute_modality_similarities(text_embs, image_embs, fused_embs, labels):
    """Compute meaningful similarities between different modalities"""
    
    results = {}
    
    # 1. Text-Image similarity (cross-modal) - Most important!
    text_image_similarity = []
    for i in range(len(text_embs)):
        sim = cosine_similarity([text_embs[i]], [image_embs[i]])[0][0]
        text_image_similarity.append(sim)
    
    text_image_similarity = np.array(text_image_similarity)
    
    results['text_image_similarity'] = {
        'values': text_image_similarity,
        'mean': np.mean(text_image_similarity),
        'std': np.std(text_image_similarity)
    }
    
    # 2. Intra-modal vs Inter-modal similarity analysis
    # Compare similarity within same modality vs across modalities
    text_intra_similarity = cosine_similarity(text_embs[:100], text_embs[:100])  # Sample for efficiency
    image_intra_similarity = cosine_similarity(image_embs[:100], image_embs[:100])
    cross_modal_similarity = cosine_similarity(text_embs[:100], image_embs[:100])
    
    # Remove diagonal for intra-modal (self-similarity is always 1)
    text_intra_mean = (np.sum(text_intra_similarity) - 100) / (100 * 99)  # Subtract diagonal
    image_intra_mean = (np.sum(image_intra_similarity) - 100) / (100 * 99)
    cross_modal_mean = np.mean(cross_modal_similarity)
    
    results['similarity_comparison'] = {
        'text_intra_similarity': text_intra_mean,
        'image_intra_similarity': image_intra_mean,
        'cross_modal_similarity': cross_modal_mean,
        'similarity_ratio': cross_modal_mean / ((text_intra_mean + image_intra_mean) / 2)
    }
    
    # 3. Modality alignment by class
    unique_labels = np.unique(labels)
    class_alignment = {}
    
    for label in unique_labels:
        mask = labels == label
        class_text = text_embs[mask]
        class_image = image_embs[mask]
        
        # Average similarity within class
        if len(class_text) > 1:
            class_sim = cosine_similarity(class_text, class_image)
            intra_class_sim = np.mean(np.diag(class_sim))
            
            class_alignment[f"class_{label}"] = {
                'text_image_alignment': intra_class_sim,
                'n_samples': len(class_text),
                'text_std': np.mean(np.std(class_text, axis=0)),  # Text embedding diversity
                'image_std': np.mean(np.std(class_image, axis=0))  # Image embedding diversity
            }
    
    results['class_alignment'] = class_alignment
    
    # 4. Correlation between modalities
    # Flatten embeddings and compute correlation
    text_flat = text_embs.flatten()
    image_flat = image_embs.flatten()
    
    # Sample to avoid memory issues
    sample_size = min(10000, len(text_flat))
    indices = np.random.choice(len(text_flat), sample_size, replace=False)
    
    correlation = np.corrcoef(text_flat[indices], image_flat[indices])[0, 1]
    results['modality_correlation'] = correlation
    
    return results

def analyze_information_complementarity(text_embs, image_embs, fused_embs, labels):
    """
    Analyze how much new information each modality adds
    """
    
    # 1. Variance analysis
    text_variance = np.var(text_embs, axis=0)
    image_variance = np.var(image_embs, axis=0)
    fused_variance = np.var(fused_embs, axis=0)
    
    # 2. PCA to see if modalities capture different information
    pca_text = PCA(n_components=10)
    pca_image = PCA(n_components=10)
    pca_fused = PCA(n_components=20)  # More components for fused
    
    pca_text.fit(text_embs)
    pca_image.fit(image_embs)
    pca_fused.fit(fused_embs)
    
    # 3. Check if fused representation captures more variance
    text_explained_var = np.sum(pca_text.explained_variance_ratio_)
    image_explained_var = np.sum(pca_image.explained_variance_ratio_)
    fused_explained_var = np.sum(pca_fused.explained_variance_ratio_)
    
    # 4. Modality dominance analysis
    text_norm = np.linalg.norm(text_embs, axis=1)
    image_norm = np.linalg.norm(image_embs, axis=1)
    modality_ratio = text_norm / (text_norm + image_norm)
    
    # 5. Information overlap analysis using CCA (Canonical Correlation Analysis)
    from sklearn.cross_decomposition import CCA
    
    # Sample for computational efficiency
    sample_size = min(1000, len(text_embs))
    indices = np.random.choice(len(text_embs), sample_size, replace=False)
    text_sample = text_embs[indices]
    image_sample = image_embs[indices]
    
    cca = CCA(n_components=1)
    cca.fit(text_sample, image_sample)
    text_c, image_c = cca.transform(text_sample, image_sample)
    canonical_corr = np.corrcoef(text_c.T, image_c.T)[0, 1]
    
    return {
        'variance_analysis': {
            'text_mean_variance': np.mean(text_variance),
            'image_mean_variance': np.mean(image_variance),
            'fused_mean_variance': np.mean(fused_variance),
            'text_total_variance': np.sum(text_variance),
            'image_total_variance': np.sum(image_variance),
            'fused_total_variance': np.sum(fused_variance)
        },
        'pca_analysis': {
            'text_explained_var': text_explained_var,
            'image_explained_var': image_explained_var,
            'fused_explained_var': fused_explained_var,
            'text_components': pca_text.explained_variance_ratio_,
            'image_components': pca_image.explained_variance_ratio_,
            'fused_components': pca_fused.explained_variance_ratio_
        },
        'modality_dominance': {
            'modality_ratio': modality_ratio,
            'text_dominant_samples': np.sum(modality_ratio > 0.6),
            'image_dominant_samples': np.sum(modality_ratio < 0.4),
            'balanced_samples': np.sum((modality_ratio >= 0.4) & (modality_ratio <= 0.6))
        },
        'information_overlap': {
            'canonical_correlation': canonical_corr,
            'shared_variance': canonical_corr ** 2  # Proportion of shared variance
        }
    }

def run_fusion_analysis(text_model, vit_model, dataloader, device, model_name="UnixCoder"):
    """
    Run complete fusion analysis
    """
    
    print(f"🔍 Analyzing {model_name} fusion embeddings...")
    
    # Extract all embeddings
    text_embs, image_embs, fused_embs, labels = analyze_fusion_embeddings(
        text_model, vit_model, dataloader, device
    )
    
    print(f"📊 Extracted embeddings: Text{text_embs.shape}, Image{image_embs.shape}, Fused{fused_embs.shape}")
    
    # Compute similarities
    similarity_results = compute_modality_similarities(text_embs, image_embs, fused_embs, labels)
    
    # Information analysis
    info_results = analyze_information_complementarity(text_embs, image_embs, fused_embs, labels)
    
    # Print insights
    print(f"\n🎯 {model_name} FUSION INSIGHTS:")
    print(f"Cross-modal similarity: {similarity_results['text_image_similarity']['mean']:.4f}")
    print(f"Canonical correlation: {info_results['information_overlap']['canonical_correlation']:.4f}")
    print(f"Shared variance: {info_results['information_overlap']['shared_variance']:.4f}")
    
    # Similarity comparison
    sim_comp = similarity_results['similarity_comparison']
    print(f"\n📈 Similarity Comparison:")
    print(f"  Text intra-modal: {sim_comp['text_intra_similarity']:.4f}")
    print(f"  Image intra-modal: {sim_comp['image_intra_similarity']:.4f}")
    print(f"  Cross-modal: {sim_comp['cross_modal_similarity']:.4f}")
    print(f"  Similarity ratio: {sim_comp['similarity_ratio']:.4f}")
    
    dominance = info_results['modality_dominance']
    print(f"\n⚖️  Modality Dominance:")
    print(f"  Text-dominant samples: {dominance['text_dominant_samples']} ({dominance['text_dominant_samples']/len(text_embs)*100:.1f}%)")
    print(f"  Image-dominant samples: {dominance['image_dominant_samples']} ({dominance['image_dominant_samples']/len(text_embs)*100:.1f}%)")
    print(f"  Balanced samples: {dominance['balanced_samples']} ({dominance['balanced_samples']/len(text_embs)*100:.1f}%)")
    
    # Interpretation
    text_image_sim = similarity_results['text_image_similarity']['mean']
    canonical_corr = info_results['information_overlap']['canonical_correlation']
    
    print(f"\n💡 INTERPRETATION:")
    if text_image_sim > 0.3 or canonical_corr > 0.5:
        print("⚠️  High alignment: Text and image embeddings capture similar information")
        print("   Visual modality may not add much new information")
    elif text_image_sim < 0.1 and canonical_corr < 0.2:
        print("✅ Low alignment: Modalities provide complementary information")
        print("   Fusion should provide significant benefits")
    else:
        print("🔄 Moderate alignment: Some information overlap with room for complementarity")
        print("   Fusion may provide moderate benefits")
    
    # Check if one modality is much stronger than the other
    text_dom_percent = dominance['text_dominant_samples']/len(text_embs)*100
    if text_dom_percent > 70:
        print(f"📝 Text modality dominates ({text_dom_percent:.1f}%) - Visual information might be ignored")
    elif text_dom_percent < 30:
        print(f"🖼️  Image modality dominates ({100-text_dom_percent:.1f}%) - Text information might be ignored")
    
    return {
        'similarity_results': similarity_results,
        'info_results': info_results,
        'embeddings': (text_embs, image_embs, fused_embs, labels),
        'model_name': model_name
    }

In [18]:
# Run fusion analysis on training data
analysis_results = run_fusion_analysis(text_model, vit_model, train_loader, DEVICE, "UnixCoder")

🔍 Analyzing UnixCoder fusion embeddings...


Extracting multi-modal embeddings: 100%|██████████| 774/774 [02:52<00:00,  4.48it/s]


📊 Extracted embeddings: Text(6190, 768), Image(6190, 768), Fused(6190, 1536)

🎯 UnixCoder FUSION INSIGHTS:
Cross-modal similarity: -0.0017
Canonical correlation: 0.9999
Shared variance: 0.9998

📈 Similarity Comparison:
  Text intra-modal: 0.2151
  Image intra-modal: 0.6564
  Cross-modal: 0.0108
  Similarity ratio: 0.0248

⚖️  Modality Dominance:
  Text-dominant samples: 5809 (93.8%)
  Image-dominant samples: 0 (0.0%)
  Balanced samples: 381 (6.2%)

💡 INTERPRETATION:
⚠️  High alignment: Text and image embeddings capture similar information
   Visual modality may not add much new information
📝 Text modality dominates (93.8%) - Visual information might be ignored


In [31]:
# Run fusion analysis on training data
analysis_results = run_fusion_analysis(text_model, vit_model, train_loader, DEVICE, "UnixCoder")

🔍 Analyzing UnixCoder fusion embeddings...


Extracting multi-modal embeddings: 100%|██████████| 774/774 [01:22<00:00,  9.40it/s]


📊 Extracted embeddings: Text(6190, 768), Image(6190, 768), Fused(6190, 1536)

🎯 UnixCoder FUSION INSIGHTS:
Cross-modal similarity: -0.0017
Canonical correlation: 0.9994
Shared variance: 0.9988

📈 Similarity Comparison:
  Text intra-modal: 0.2127
  Image intra-modal: 0.6451
  Cross-modal: 0.0102
  Similarity ratio: 0.0239

⚖️  Modality Dominance:
  Text-dominant samples: 5809 (93.8%)
  Image-dominant samples: 0 (0.0%)
  Balanced samples: 381 (6.2%)

💡 INTERPRETATION:
⚠️  High alignment: Text and image embeddings capture similar information
   Visual modality may not add much new information
📝 Text modality dominates (93.8%) - Visual information might be ignored


**TESTING**

In [32]:
from torch.utils.data import DataLoader
from tqdm import tqdm

def test_on_dataset(text_dir, image_dir, tokenizer, image_processor, fusion_model, batch_size=4):
    """🧪 Test the fusion model on a single dataset"""

    dataset = FusionDataset(text_dir, image_dir, tokenizer, image_processor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    fusion_model.eval()
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Testing Dataset 📝"):
            input_ids, attention_mask, image_tensor, labels = batch

            # Move text to device
            input_ids = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
            labels = labels.to(DEVICE)

            # Move image tensors to device
            for k in image_tensor:
                image_tensor[k] = image_tensor[k].to(DEVICE)

            # Forward pass
            logits = fusion_model(input_ids=input_ids, attention_mask=attention_mask, image_tensor=image_tensor)
            preds = torch.argmax(logits, dim=1)

            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

    accuracy = total_correct / total_samples
    print(f"✅ Dataset Accuracy: {accuracy*100:.2f}% 🎉")
    return accuracy

In [34]:
FUSION_CKPT = "../checkpoints/early_fusion_unixcoder_nn/fusion_norm_epoch_3.pt"
fusion_model = FusionClassifier(text_model, vit_model)
fusion_model.to(DEVICE)
# Load checkpoint
state_dict = torch.load(FUSION_CKPT, map_location=DEVICE)
fusion_model.load_state_dict(state_dict)
fusion_model.eval()

FusionClassifier(
  (text_model): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(51416, 768, padding_idx=1)
      (position_embeddings): Embedding(1026, 768, padding_idx=1)
      (token_type_embeddings): Embedding(10, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (Layer

In [16]:
for i in range(10):
    print(f"Test_{i}")
    test_on_dataset(f"../Text_Files/Test_{i}", f"../snapshots/Test_{i}", tokenizer, image_processor, fusion_model, batch_size=4)

Test_0


Testing Dataset 📝: 100%|██████████| 251/251 [00:13<00:00, 18.05it/s]


✅ Dataset Accuracy: 84.43% 🎉
Test_1


Testing Dataset 📝: 100%|██████████| 251/251 [00:13<00:00, 18.08it/s]


✅ Dataset Accuracy: 81.44% 🎉
Test_2


...

✅ Dataset Accuracy: 73.75% 🎉
Test_8


Testing Dataset 📝: 100%|██████████| 251/251 [00:13<00:00, 18.08it/s]


✅ Dataset Accuracy: 79.64% 🎉
Test_9


Testing Dataset 📝: 100%|██████████| 251/251 [00:13<00:00, 18.12it/s]

✅ Dataset Accuracy: 73.75% 🎉


In [35]:
for i in range(10):
    print(f"Test_{i}")
    test_on_dataset(f"../Text_Files/Test_{i}", f"../snapshots/Test_{i}", tokenizer, image_processor, fusion_model, batch_size=4)

Test_0


Testing Dataset 📝: 100%|██████████| 251/251 [00:13<00:00, 17.96it/s]


✅ Dataset Accuracy: 84.83% 🎉
Test_1


Testing Dataset 📝: 100%|██████████| 251/251 [00:13<00:00, 18.12it/s]


✅ Dataset Accuracy: 81.14% 🎉
Test_2


...

✅ Dataset Accuracy: 73.15% 🎉
Test_8


Testing Dataset 📝: 100%|██████████| 251/251 [00:13<00:00, 18.00it/s]


✅ Dataset Accuracy: 78.94% 🎉
Test_9


Testing Dataset 📝: 100%|██████████| 251/251 [00:13<00:00, 18.06it/s]

✅ Dataset Accuracy: 73.15% 🎉
